# Next Word Prediction Using Transformer


In [1]:
import os
import re
import pickle
import datetime
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split


def pad_sequences(sequences, maxlen, padding="pre"):
    padded = np.zeros((len(sequences), maxlen), dtype=np.int64)
    for i, seq in enumerate(sequences):
        seq = list(seq)
        if len(seq) > maxlen:
            seq = seq[-maxlen:]
        if not seq:
            continue
        if padding == "pre":
            padded[i, -len(seq) :] = seq
        else:
            padded[i, : len(seq)] = seq
    return padded

In [2]:
# Retrain the model
# IS_TRAIN_MODE = True

# Load the model from save
IS_TRAIN_MODE = False

In [3]:
# Load the data

baseDir = os.getcwd()
# curDir = os.path.join(baseDir, "T10 - LLM", "S02 - Recurrent")
curDir = baseDir
filePath = os.path.join(curDir, "hamlet.txt")
print(filePath)

if not os.path.exists(filePath):
    import nltk

    nltk.download("gutenberg")
    from nltk.corpus import gutenberg

    data = gutenberg.raw("shakespeare-hamlet.txt")
    with open(filePath, "w") as file:
        file.write(data)


with open(filePath) as file:
    text = file.read().lower()

c:\Users\admin\Coding\ai-technique-68\codes\src\T08_llm\S03 - Transformer\hamlet.txt


In [4]:
# Show the first 10 lines
for idx, line in enumerate(text.split("\n")):
    print(line)
    if idx > 10:
        break

[the tragedie of hamlet by william shakespeare 1599]


actus primus. scoena prima.

enter barnardo and francisco two centinels.

  barnardo. who's there?
  fran. nay answer me: stand & vnfold
your selfe

   bar. long liue the king


In [5]:
# Create a simple tokenizer
class SimpleTokenizer:
    def __init__(self):
        self.word_index = {}
        self.index_word = {}

    def _tokenize(self, text):
        return re.findall(r"\b\w+\b", text.lower())

    def fit_on_texts(self, texts):
        vocab = []
        seen = set()
        for text in texts:
            for token in self._tokenize(text):
                if token not in seen:
                    seen.add(token)
                    vocab.append(token)

        self.word_index = {word: idx + 1 for idx, word in enumerate(vocab)}
        self.index_word = {idx: word for word, idx in self.word_index.items()}

    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            tokens = self._tokenize(text)
            seq = [
                self.word_index[token] for token in tokens if token in self.word_index
            ]
            sequences.append(seq)
        return sequences


tokenizer = SimpleTokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1
print(total_words)

4702


In [6]:
# Show the first 20 words as a dictionary
for idx, (k, v) in enumerate(tokenizer.word_index.items()):
    print(f"{k:15s} -> {v:5d}")
    if idx > 20:
        break

the             ->     1
tragedie        ->     2
of              ->     3
hamlet          ->     4
by              ->     5
william         ->     6
shakespeare     ->     7
1599            ->     8
actus           ->     9
primus          ->    10
scoena          ->    11
prima           ->    12
enter           ->    13
barnardo        ->    14
and             ->    15
francisco       ->    16
two             ->    17
centinels       ->    18
who             ->    19
s               ->    20
there           ->    21
fran            ->    22


In [7]:
# Test Out of Vocabulary Word
# The tokenizer will ignore the word "CMU" as it is not in the vocabulary
print(tokenizer.texts_to_sequences(["CMU great"]))

[[805]]


In [8]:
# Creating input-sequence

input_sequences = []
for line in text.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[: i + 1]
        input_sequences.append(n_gram_sequence)

input_sequences[:5]

[[1, 2], [1, 2, 3], [1, 2, 3, 4], [1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6]]

In [9]:
max_sequence_length = max([len(x) for x in input_sequences])
print(max_sequence_length)

14


In [10]:
# Pad sequences
input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_sequence_length, padding="pre")
)
input_sequences

array([[   0,    0,    0, ...,    0,    1,    2],
       [   0,    0,    0, ...,    1,    2,    3],
       [   0,    0,    0, ...,    2,    3,    4],
       ...,
       [   0,    0,    0, ...,    3,    4, 2037],
       [   0,    0,    0, ...,    4, 2037,    3],
       [   0,    0,    0, ..., 2037,    3,  216]], shape=(26305, 14))

In [11]:
# Create predictors and labels
X, yt = input_sequences[:, :-1], input_sequences[:, -1]

In [12]:
print(X.shape)
print(X[:5])

(26305, 13)
[[0 0 0 0 0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 0 0 0 1 2]
 [0 0 0 0 0 0 0 0 0 0 1 2 3]
 [0 0 0 0 0 0 0 0 0 1 2 3 4]
 [0 0 0 0 0 0 0 0 1 2 3 4 5]]


In [13]:
yt[:5]

array([2, 3, 4, 5, 6])

In [14]:
# Labels for PyTorch CrossEntropyLoss (class indices, not one-hot)
y = yt.astype(np.int64)
print(y.shape)
print(y[:10])

(26305,)
[ 2  3  4  5  6  7  8 10 11 12]


In [15]:
# Split the data
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [16]:
# Create the model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


class NextWordTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        max_len,
        d_model=128,
        nhead=8,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.1,
    ):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.position_embedding = nn.Embedding(max_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            enable_nested_tensor=False,
        )
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        batch_size, seq_len = x.size()
        pos_idx = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, -1)

        tok = self.token_embedding(x)
        pos = self.position_embedding(pos_idx)
        h = tok + pos

        key_padding_mask = x.eq(0)
        h = self.transformer(h, src_key_padding_mask=key_padding_mask)
        h = self.norm(h)

        last_hidden = h[:, -1, :]
        return self.fc(last_hidden)


model = NextWordTransformer(
    vocab_size=total_words,
    max_len=max_sequence_length - 1,
    d_model=128,
    nhead=8,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1,
).to(device)

Using device: cpu


In [17]:
dummy_x = torch.randint(0, total_words, (100, max_sequence_length - 1)).to(device)
dummy_out = model(dummy_x)
print(dummy_x.shape)
print(dummy_out.shape)

torch.Size([100, 13])
torch.Size([100, 4702])


In [18]:
from torchinfo import summary

input_data = torch.randint(
    low=1,
    high=total_words,
    size=(100, max_sequence_length - 1),
    dtype=torch.long,
    device=device,
 )
summary(model, input_data=input_data, device=str(device))

Layer (type:depth-idx)                        Output Shape              Param #
NextWordTransformer                           [100, 4702]               --
├─Embedding: 1-1                              [100, 13, 128]            601,856
├─Embedding: 1-2                              [100, 13, 128]            1,664
├─TransformerEncoder: 1-3                     [100, 13, 128]            --
│    └─ModuleList: 2-1                        --                        --
│    │    └─TransformerEncoderLayer: 3-1      [100, 13, 128]            132,480
│    │    └─TransformerEncoderLayer: 3-2      [100, 13, 128]            132,480
├─LayerNorm: 1-4                              [100, 13, 128]            256
├─Linear: 1-5                                 [100, 4702]               606,558
Total params: 1,475,294
Trainable params: 1,475,294
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 134.32
Input size (MB): 0.01
Forward/backward pass size (MB): 21.07
Params size (MB): 5.37
Estimated Total Siz

In [19]:
# Train the model

if IS_TRAIN_MODE:
    x_train_t = torch.tensor(x_train, dtype=torch.long)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    x_test_t = torch.tensor(x_test, dtype=torch.long)
    y_test_t = torch.tensor(y_test, dtype=torch.long)

    train_ds = torch.utils.data.TensorDataset(x_train_t, y_train_t)
    test_ds = torch.utils.data.TensorDataset(x_test_t, y_test_t)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=256, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=256, shuffle=False)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    epochs = 40
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * xb.size(0)
            preds = torch.argmax(logits, dim=1)
            train_correct += (preds == yb).sum().item()
            train_total += xb.size(0)

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
                preds = torch.argmax(logits, dim=1)
                val_correct += (preds == yb).sum().item()
                val_total += xb.size(0)

        train_loss /= max(train_total, 1)
        train_acc = train_correct / max(train_total, 1)
        val_loss /= max(val_total, 1)
        val_acc = val_correct / max(val_total, 1)

        print(
            f"Epoch {epoch + 1:02d}/{epochs} - "
            f"loss: {train_loss:.4f} - acc: {train_acc:.4f} - "
            f"val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}"
        )

    # Save the model
    dateTime = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    model_path = os.path.join(curDir, f"transformer-model-{dateTime}.pt")
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "vocab_size": total_words,
            "max_sequence_length": max_sequence_length,
            "d_model": 128,
            "nhead": 8,
            "num_layers": 2,
            "dim_feedforward": 256,
            "dropout": 0.1,
        },
        model_path,
    )
    print(f"Saved model to: {model_path}")

    # Save tokenizer
    with open(os.path.join(curDir, "tokenizer.pkl"), "wb") as handle:
        pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [20]:
# Load the model
if not IS_TRAIN_MODE:
    model_files = sorted(
        [f for f in os.listdir(curDir) if f.startswith("transformer-model-") and f.endswith(".pt")]
    )
    if not model_files:
        raise FileNotFoundError(
            "No transformer checkpoint found. Train first or place a transformer-model-*.pt file in curDir."
        )

    checkpoint_path = os.path.join(curDir, model_files[-1])
    print(f"Loading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model = NextWordTransformer(
        vocab_size=checkpoint.get("vocab_size", total_words),
        max_len=checkpoint.get("max_sequence_length", max_sequence_length) - 1,
        d_model=checkpoint.get("d_model", 128),
        nhead=checkpoint.get("nhead", 8),
        num_layers=checkpoint.get("num_layers", 2),
        dim_feedforward=checkpoint.get("dim_feedforward", 256),
        dropout=checkpoint.get("dropout", 0.1),
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    max_sequence_length = checkpoint.get("max_sequence_length", max_sequence_length)
    print(model)

Loading checkpoint: c:\Users\admin\Coding\ai-technique-68\codes\src\T08_llm\S03 - Transformer\transformer-model-20260311-042250.pt
NextWordTransformer(
  (token_embedding): Embedding(4702, 128, padding_idx=0)
  (position_embedding): Embedding(13, 128)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((128,), eps=1e-05, 

In [21]:
# Function to predict next word


def predict_next_word(model, tokenizer, text, max_sequence_length):
    token_list = tokenizer.texts_to_sequences([text])[0]
    if len(token_list) >= max_sequence_length:
        token_list = token_list[-max_sequence_length:]

    token_list = pad_sequences(
        [token_list], maxlen=max_sequence_length - 1, padding="pre"
    )

    input_tensor = torch.tensor(token_list, dtype=torch.long, device=device)

    model.eval()
    with torch.no_grad():
        logits = model(input_tensor)
        predicted_word_index = int(torch.argmax(logits, dim=1).item())

    return tokenizer.index_word.get(predicted_word_index)

In [22]:
# Predict next word

input_text = "With mirth the king"
print(f"Input text: {input_text}")

token_list = tokenizer.texts_to_sequences([input_text])[0]
print(f"Padded token list: {token_list}")

token_list = pad_sequences([token_list], maxlen=max_sequence_length - 1, padding="pre")
print(f"Token list: {token_list}")

input_tensor = torch.tensor(token_list, dtype=torch.long, device=device)
model.eval()
with torch.no_grad():
    logits = model(input_tensor)
    probs = torch.softmax(logits, dim=1).cpu().numpy()

print(f"Predicted probabilities shape: {probs.shape}")

predicted_word_index = int(np.argmax(probs, axis=1)[0])
print(f"Predicted word index: {predicted_word_index}")

predicted_word = tokenizer.index_word.get(predicted_word_index)
print(f"Predicted next word: {predicted_word}")

Input text: With mirth the king
Padded token list: [136, 561, 1, 33]
Token list: [[  0   0   0   0   0   0   0   0   0 136 561   1  33]]
Predicted probabilities shape: (1, 4702)
Predicted word index: 15
Predicted next word: and


In [23]:
# Generate text
input_text = "I am"

textStr = input_text
print(textStr, end=" ")
for i in range(1, 300):
    next_word = predict_next_word(model, tokenizer, textStr, max_sequence_length)
    if next_word is None:
        break
    print(next_word, end=" ")
    if i % 20 == 0:
        print("\n", end="")
    textStr = textStr + " " + next_word

I am iustly kill d and lacke gall d to be all to be your pratlings of grace whereof tend what he 
to bee feare is guilt d such an king is too late with beating one come some eight yeare or 
second his head shake father to be done to morrow as proper me a sodaine art more ingag d vpon 
your honour d as you do to heauen and making loue is mine owne eyes into them let s follow 
tis not fit thus to obey him and out his wits a fellow in his face and obedience marke this 
you to be my offer it is this to his ponderous do thinke in t which haue lodg o re 
swaies his inclination you trippingly such a dreames d all his visage on his backe to do him know you 
call vp the staires into the lobby it not seemes it one on your lippes i pray you your selfe 
in the hall if it see another question next day night night with your water had not art thou wouldest 
d to a nunnery go farre of question with beating comes to his face as damn her owne defence let 
vertue it is proofe much throwing come thing in 